# Dev / Test / Prod Environments (Serverless Model)

This notebook introduces environment management and testing patterns for local AI applications.

Topics:
- Environment-specific `.env` files
- Development, Test, and Production configurations
- In-memory vs file-backed SQLite
- Pytest fixtures
- Test data seeding
- Test teardown
- Local evaluation workflows

The focus is on lightweight local/serverless development before introducing cloud deployments.


## Learning Goals

By the end of this notebook you should be able to:

1. Manage environment-specific configuration.
2. Separate dev, test, and production settings.
3. Use SQLite differently across environments.
4. Seed databases for testing.
5. Clean up test artifacts automatically.
6. Build repeatable local evaluation workflows.


In [1]:
%pip install -q python-dotenv pytest sqlmodel


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Environment Strategy

A common layout:

```text
project/
│
├── .env.dev
├── .env.test
├── .env.prod
│
├── data/
│   ├── dev.db
│   ├── test.db
│   └── prod.db
│
├── tests/
└── src/
```

Each environment has its own settings.


In [2]:
from dotenv import load_dotenv
import os

APP_ENV = os.getenv("APP_ENV", "dev")

env_map = {
    "dev": ".env.dev",
    "test": ".env.test",
    "prod": ".env.prod"
}

load_dotenv(env_map.get(APP_ENV, ".env.dev"))

print("Environment:", APP_ENV)


Environment: dev


# Example Environment Files

## .env.dev

```env
APP_ENV=dev
DB_PATH=data/dev.db
LOG_LEVEL=DEBUG
```

## .env.test

```env
APP_ENV=test
DB_PATH=:memory:
LOG_LEVEL=INFO
```

## .env.prod

```env
APP_ENV=prod
DB_PATH=data/prod.db
LOG_LEVEL=WARNING
```


# SQLite Lifecycle

Development:
- Persistent database file

Testing:
- In-memory database

Production:
- Persistent production database

This keeps tests isolated and fast.


In [3]:
import sqlite3

# Test database
test_conn = sqlite3.connect(":memory:")

test_conn.execute(
    '''
    CREATE TABLE prompts(
        id INTEGER PRIMARY KEY,
        prompt TEXT
    )
    '''
)

print("In-memory database ready")


In-memory database ready


In [4]:
# Production-style database

prod_conn = sqlite3.connect("prod.db")

prod_conn.execute(
    '''
    CREATE TABLE IF NOT EXISTS prompts(
        id INTEGER PRIMARY KEY,
        prompt TEXT
    )
    '''
)

prod_conn.commit()

print("Persistent database ready")


Persistent database ready


# Environment-Aware Database Factory

Centralize connection creation.


In [5]:
import sqlite3
import os

def get_db_connection():

    db_path = os.getenv("DB_PATH", "dev.db")

    return sqlite3.connect(db_path)

conn = get_db_connection()

print(conn)


# Why In-Memory Databases for Tests?

Benefits:

- Fast
- No cleanup required
- Isolated
- Repeatable
- No accidental production data modification


# Pytest Fixtures

Fixtures create reusable setup and teardown logic.

Typical flow:

Setup
    ↓
Seed Data
    ↓
Run Tests
    ↓
Cleanup


In [6]:
# tests/conftest.py

import pytest
import sqlite3

@pytest.fixture
def db_connection():

    conn = sqlite3.connect(":memory:")

    conn.execute(
        '''
        CREATE TABLE prompts(
            id INTEGER PRIMARY KEY,
            prompt TEXT
        )
        '''
    )

    yield conn

    conn.close()


# Seeding Test Data

Seed realistic data before running tests.


In [7]:
def seed_prompts(conn):

    prompts = [
        ("What is RAG?",),
        ("Explain vector databases",),
        ("What is LangGraph?",)
    ]

    conn.executemany(
        "INSERT INTO prompts(prompt) VALUES (?)",
        prompts
    )

    conn.commit()


In [8]:
# Example test

def test_prompt_count(db_connection):

    seed_prompts(db_connection)

    cursor = db_connection.execute(
        "SELECT COUNT(*) FROM prompts"
    )

    count = cursor.fetchone()[0]

    assert count == 3


# File-Based Test Databases

Sometimes you need persistence during tests.

Example:

```text
tests/
└── test.db
```

Create it before the test and delete it afterward.


In [9]:
from pathlib import Path

TEST_DB = Path("test.db")

if TEST_DB.exists():
    TEST_DB.unlink()

print("Clean test state")


Clean test state


In [10]:
# Fixture with cleanup

import pytest
import sqlite3

@pytest.fixture
def file_db():

    conn = sqlite3.connect("test.db")

    yield conn

    conn.close()

    Path("test.db").unlink(missing_ok=True)


# Mock Vector Store Seeding

Before learning Chroma and RAG evaluation patterns,
you can simulate vector records.


In [11]:
mock_vectors = [
    {
        "id": "1",
        "text": "RAG combines retrieval and generation."
    },
    {
        "id": "2",
        "text": "Chroma stores embeddings."
    }
]

print(mock_vectors)


[{'id': '1', 'text': 'RAG combines retrieval and generation.'}, {'id': '2', 'text': 'Chroma stores embeddings.'}]


# Evaluation Workflow

Simple local workflow:

1. Seed relational data
2. Seed vector data
3. Run retrieval tests
4. Verify outputs
5. Cleanup

This pattern scales nicely into RAG evaluations later.


In [12]:
def evaluate():

    expected = "RAG"

    response = "RAG combines retrieval and generation."

    assert expected in response

    return "Evaluation Passed"

print(evaluate())


Evaluation Passed


# CI/CD Perspective

A typical pipeline:

Developer Push
        ↓
Unit Tests
        ↓
Seed Test Data
        ↓
Evaluation Tests
        ↓
Cleanup
        ↓
Deploy

LangSmith CI/CD examples use a similar philosophy of automated validation before deployment.


# Recommended Local Strategy

Development:
- dev.db

Testing:
- :memory:

Production:
- prod.db

Benefits:
- Simplicity
- Fast testing
- Safe isolation
- Easy debugging


# Mini Exercise

1. Create `.env.dev`, `.env.test`, and `.env.prod`.
2. Build a database factory.
3. Add a token_usage table.
4. Create pytest fixtures.
5. Seed sample data.
6. Verify row counts.
7. Clean up test artifacts.


# Key Takeaways

- Keep configuration environment-specific.
- Use python-dotenv for loading settings.
- Use `:memory:` SQLite for tests.
- Use file-backed SQLite for persistent environments.
- Use pytest fixtures for setup and teardown.
- Seed realistic test data.
- Remove test artifacts automatically.
- Build repeatable evaluation workflows before moving into RAG and agents.
